In [ ]:
!pip install scikit-learn --quiet

In [ ]:
student_answer = """au) Su) oen ee ae eee eee ee maa\nCoperation flood] were set ve by the government, bot this led to\n| development 10. concentrated areas, 2\na 3) Farmers were given crop insyrapedaabinst failure of crersi?\n( case of ‘droughts, fleeds. Fires cic. Establisnment of grameen\nj anks , cooperatives and banks that provicted loans at reasonable\n( totes of \\nterest: poten abies aearh\n| 3) Kisan Credit Cork, PALS ( Personal _acctdbet Insurance Scheme},\n| fenumerotive prices, Special weethec bulletins Special. programmes\n1 on Tvand radio channels were also eee eee ee we\n__|€xpleHation_by mictelemen and specolaters.
"""

print(f"Word count: {len(student_answer.split())}")

In [ ]:
TEACHER_ANSWER_FILE = "teacher_answer.txt"

with open(TEACHER_ANSWER_FILE, "r", encoding="utf-8") as f:
    teacher_answer = f.read().strip()

print(f"Word count: {len(teacher_answer.split())}")
print("\nPreview:")
print(teacher_answer[:300], "..." if len(teacher_answer) > 300 else "")

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def clean(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def compute_similarity(student_text, teacher_text):
    s = clean(student_text)
    t = clean(teacher_text)

    vectorizer = TfidfVectorizer()
    tfidf      = vectorizer.fit_transform([s, t])
    sim        = cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]
    return round(float(sim), 4)


def similarity_to_marks(sim, total=10):
    if sim >= 0.85: marks = 9.0 + (sim - 0.85) / 0.15
    elif sim >= 0.70: marks = 7.0 + (sim - 0.70) / 0.15 * 2
    elif sim >= 0.55: marks = 5.0 + (sim - 0.55) / 0.15 * 2
    elif sim >= 0.40: marks = 3.0 + (sim - 0.40) / 0.15 * 2
    else: marks = max(0.0, sim / 0.40 * 3)
    return round(min(marks, total), 1)


def grade(marks, total=10):
    r = marks / total
    if r >= 0.85: return "Excellent"
    if r >= 0.70: return "Good"
    if r >= 0.55: return "Average"
    if r >= 0.40: return "Below Average"
    return "Poor"


similarity = compute_similarity(student_answer, teacher_answer)
marks      = similarity_to_marks(similarity)
grade_str  = grade(marks)

print("Scoring done")

In [ ]:
sep = "═" * 55
print(sep)
print("MARKING REPORT")
print(sep)
print(f"  Similarity Score : {similarity:.4f}  (out of 1.0)")
print(f"  Marks Awarded    : {marks} / 10")
print(f"  Grade            : {grade_str}")
print(sep)

from sklearn.feature_extraction.text import CountVectorizer

student_words = set(clean(student_answer).split())
teacher_words = set(clean(teacher_answer).split())

stopwords = {'and','the','of','in','to','a','is','are','was','were',
             'by','on','at','for','with','this','that','it','be','an',
             'also','but','or','not','from','as','up','set','also'}
student_kw = student_words - stopwords
teacher_kw = teacher_words - stopwords

matched   = student_kw & teacher_kw
missed    = teacher_kw - student_kw

print(f"\n  Keywords matched : {len(matched)} → {', '.join(sorted(matched)[:15])}")
print(f"  Keywords missed  : {len(missed)}  → {', '.join(sorted(missed)[:15])}")
print(sep)

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import requests
from PIL import Image
from huggingface_hub import login


processor = TrOCRProcessor.from_pretrained("microsoft/trocr-large-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-large-handwritten")

Loading weights: 100%|██████████| 635/635 [00:00<00:00, 49691.84it/s]
The tied weights mapping and config for this model specifies to tie decoder.model.decoder.embed_tokens.weight to decoder.output_projection.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-large-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.weight | MISSING | 
encoder.pooler.dense.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
import cv2
import torch


IMAGE_PATH = "sheet.png"
image = cv2.imread(IMAGE_PATH)
if image is None:
    raise FileNotFoundError(f"Could not load: {IMAGE_PATH}")

image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

pixel_values = processor(image, return_tensors="pt").pixel_values
generated_ids = model.generate(pixel_values)
print(generated_ids)
generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(generated_text,"generated_text")

tensor([[   2, 1646, 5379, 5549,    2]])
1962 63 generated_text


In [13]:
pip install paddleocr paddlepaddle

  Obtaining dependency information for paddleocr from https://files.pythonhosted.org/packages/32/25/75a7aa8409d9a6ece6bd7ff9a896a25b75c5a78b2948d2119ab7a99db2f2/paddleocr-3.4.0-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 978.5 kB/s eta 0:00:00a 0:00:01
  Obtaining dependency information for paddlepaddle from https://files.pythonhosted.org/packages/5e/02/854d122760406d8024837112d91ff509e7820d763ac0a5002c4c2812f832/paddlepaddle-3.3.1-cp311-cp311-macosx_11_0_arm64.whl.metadata
  Obtaining dependency information for paddlex[ocr-core]<3.5.0,>=3.4.0 from https://files.pythonhosted.org/packages/10/39/8fd9fd72ae51e1a0b21cb5596f1125f42286d688cf28797f883b42f2c8ce/paddlex-3.4.3-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 kB 3.3 MB/s eta 0:00:00
  Obtaining dependency information for opt-einsum==3.3.0 from https://files.pythonhosted.org/packages/bc/19/404708a7e54ad2798907210462fd950c3442ea51acc8790f3da48d2bee8b/opt_einsu

In [6]:
from paddleocr import PaddleOCR

ocr = PaddleOCR(lang="en")


Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/ritikraj/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/ritikraj/.paddlex/official_models/UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/ritikraj/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/ritikraj/.paddlex/official_models/PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/Users/ritikraj/.pad

In [ ]:
import cv2
# image = cv2.imread("sheet.png")

# if image is None:
#     raise ValueError("Image not loaded. Check file path.")
# else:
#     print(type(image))

result = ocr.ocr("/Users/ritikraj/automated_edtech_grading/sheet.png")

for line in result:
    print(line[1][0])

In [10]:
import os
print(os.getcwd())

/Users/ritikraj/automated_edtech_grading
